In [ ]:
from google.colab import files
uploaded = files.upload()

Saving processed_nlp_data.csv to processed_nlp_data.csv


In [ ]:
import pandas as pd

df = pd.read_csv('processed_nlp_data.csv')

df.head()

,Customer Age,Customer Gender,Product Purchased,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,...,purchase_day,desc_length,subject_length,cleaned_description,cleaned_subject,final_description,final_subject,lemmatized_description,lemmatized_subject,final_text
0,32,Other,GoPro Hero,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,...,22,284,13,im having an issue with the productpurchased p...,product setup,im issue productpurchased please assist billin...,product setup,im issue productpurchased please assist billin...,product setup,product setup im issue productpurchased please...
1,42,Female,LG Smart TV,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,...,22,282,24,im having an issue with the productpurchased p...,peripheral compatibility,im issue productpurchased please assist need c...,peripheral compatibility,im issue productpurchased please assist need c...,peripheral compatibility,peripheral compatibility im issue productpurch...
2,48,Other,Dell XPS,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,...,14,275,15,im facing a problem with my productpurchased t...,network problem,im facing problem productpurchased productpurc...,network problem,im facing problem productpurchased productpurc...,network problem,network problem im facing problem productpurch...
3,27,Female,Microsoft Office,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,...,13,262,14,im having an issue with the productpurchased p...,account access,im issue productpurchased please assist proble...,account access,im issue productpurchased please assist proble...,account access,account access im issue productpurchased pleas...
4,67,Female,Autodesk AutoCAD,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,...,4,333,9,im having an issue with the productpurchased p...,data loss,im issue productpurchased please assist note s...,data loss,im issue productpurchased please assist note s...,data loss,data loss im issue productpurchased please ass...


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans

In [ ]:
df.columns

Index(['Customer Age', 'Customer Gender', 'Product Purchased', 'Ticket Type',
       'Ticket Subject', 'Ticket Description', 'Ticket Status', 'Resolution',
       'Ticket Priority', 'Ticket Channel', 'First Response Time',
       'Time to Resolution', 'Customer Satisfaction Rating', 'purchase_year',
       'purchase_month', 'purchase_day', 'desc_length', 'subject_length',
       'cleaned_description', 'cleaned_subject', 'final_description',
       'final_subject', 'lemmatized_description', 'lemmatized_subject',
       'final_text'],
      dtype='object')

In [ ]:
X_text = df['final_text']
y = df['Time to Resolution']

In [ ]:
y.head()

,Time to Resolution
0,NaN
1,NaN
2,2023-06-01 18:05:38
3,2023-06-01 01:57:40
4,2023-06-01 19:53:42


In [ ]:
# Convert to datetime
df['Time to Resolution'] = pd.to_datetime(df['Time to Resolution'], errors='coerce')
df['First Response Time'] = pd.to_datetime(df['First Response Time'], errors='coerce')

# Create resolution time in HOURS
df['Resolution_Hours'] = (df['Time to Resolution'] - df['First Response Time']).dt.total_seconds() / 3600

# Remove missing values
df = df.dropna(subset=['Resolution_Hours'])

# Final target
X_text = df['final_text']
y = df['Resolution_Hours']

In [ ]:
y.head()

,Resolution_Hours
2,6.850000
3,-5.533333
4,19.683333
10,-17.916667
11,-2.633333


In [ ]:
# Remove negative values
df = df[df['Resolution_Hours'] >= 0]

# Reset again
X_text = df['final_text']
y = df['Resolution_Hours']

In [ ]:
y.head()

,Resolution_Hours
2,6.850000
4,19.683333
19,19.716667
28,6.766667
29,17.483333


In [ ]:
tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(X_text)

In [ ]:
X.shape

(1404, 2338)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
X_train.shape , X_test.shape

((1123, 2338), (281, 2338))

In [ ]:
model = LinearRegression()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [ ]:
mse = mean_squared_error(y_test, y_pred)

print("MSE:", mse)

MSE: 681.8185417617887


In [ ]:
import numpy as np

rmse = np.sqrt(mse)
print("RMSE:", rmse)

RMSE: 26.111655285749094


In [ ]:
 from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_pred)
print("R2 Score:", r2)

R2 Score: -19.47514614237781


In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)

clusters = kmeans.fit_predict(X)

df['Cluster'] = clusters

In [ ]:
print("Inertia:", kmeans.inertia_)

Inertia: 1257.5382345499522


In [ ]:
df['Cluster'].value_counts()

,count
Cluster,
2,1033
1,213
0,158


In [ ]:
df[['final_text' , 'Cluster']].head(10)

,final_text,Cluster
2,network problem im facing problem productpurch...,2
4,data loss im issue productpurchased please ass...,1
19,software bug im issue productpurchased please ...,2
28,product recommendation im issue productpurchas...,2
29,cancellation request im issue productpurchased...,2
31,product compatibility im issue productpurchase...,2
33,hardware issue im issue productpurchased pleas...,2
46,network problem im issue productpurchased plea...,0
52,network problem im issue productpurchased plea...,0
53,delivery problem im issue productpurchased ple...,2


In [ ]:
import joblib

joblib.dump(model, "model.pkl")

['model.pkl']